In [ ]:
"""
Environment Setup Module.
Run this cell first to install all required libraries.
"""
!pip install qiskit[visualization] qiskit-aer qiskit-ibm-runtime matplotlib scipy pylatexenc

import math, itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import Estimator, Sampler
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE.")

---
## Exercise 1: QUBO Construction and Classical Baseline

**Asset universe:** AAPL, MSFT, JPM, XOM — annualised data 2020–2024.

**The core decision:** we must choose exactly $K=2$ assets from 4. The objective is:
$$\min_{x \in \{0,1\}^4}\; \lambda\, x^T \Sigma\, x - \boldsymbol{\mu}^T x \quad\text{s.t.}\quad \sum_i x_i = K$$

**Why convert to QUBO?** The quantum hardware only knows how to minimise Ising-type Hamiltonians ($\sum Z_iZ_j + \sum Z_i$). The QUBO matrix $Q$ is the intermediate step: it captures the quadratic objective in a form that maps directly to Pauli operators via $x_i = (1-Z_i)/2$.

**Why the penalty term $\lambda_\text{pen}(\sum_i x_i - K)^2$?**  
This converts the hard equality constraint $\sum x_i = K$ into a soft penalty added to the objective. Any solution that selects the wrong number of assets incurs a large extra cost, so the quantum circuit learns to avoid infeasible states. We need $\lambda_\text{pen}$ large enough that no infeasible solution is ever cheaper than the worst feasible one.

**Why establish a classical brute-force baseline?**  
For $n=4$ we can enumerate all $\binom{4}{2} = 6$ portfolios exhaustively in microseconds. This gives us the **ground truth** against which to benchmark QAOA. Without it we would not know if the quantum algorithm is finding the right answer.

**What you will see:** AAPL+MSFT wins at $\lambda=1$ because their high expected returns outweigh their correlation penalty. As $\lambda$ increases, the optimal portfolio shifts toward lower-correlation pairs (AAPL+XOM, then JPM+XOM).


In [ ]:
"""
QUBO Construction Module.
Builds the portfolio QUBO matrix from historical asset data and
enumerates all feasible portfolios to establish a classical baseline.
"""

# ── STUDENT EXPERIMENT ZONE ────────────────────────────────────────────────
LAM     = 1.0    # Risk-aversion parameter (try 0.5, 2.0, 5.0)
LAM_PEN = 3.0    # Budget penalty strength
K       = 2      # Number of assets to select
# ──────────────────────────────────────────────────────────────────────────

TICKERS = ['AAPL', 'MSFT', 'JPM', 'XOM']
N       = len(TICKERS)

# Annualised expected returns (2020-2024)
MU = np.array([0.340, 0.270, 0.150, 0.220])

# Annualised covariance matrix (2020-2024)
# Computed from daily return series; AAPL-MSFT highly correlated (tech),
# XOM (energy) least correlated with the group.
SIGMA = np.array([
    [0.0841, 0.0588, 0.0313, 0.0203],
    [0.0588, 0.0729, 0.0272, 0.0170],
    [0.0313, 0.0272, 0.0576, 0.0294],
    [0.0203, 0.0170, 0.0294, 0.1225]
])

print("Asset Data Summary")
print("-" * 55)
print(f"{'Ticker':>8}  {'μ (return)':>12}  {'σ (risk)':>10}  {'Sharpe':>8}")
print("-" * 55)
for i, t in enumerate(TICKERS):
    sigma_i = math.sqrt(SIGMA[i,i])
    print(f"  {t:>6}  {MU[i]*100:>10.1f}%  {sigma_i*100:>8.1f}%  "
          f"{MU[i]/sigma_i:>8.2f}")

print("\nCorrelation Matrix:")
corr = np.zeros((N,N))
for i in range(N):
    for j in range(N):
        corr[i,j] = SIGMA[i,j] / math.sqrt(SIGMA[i,i]*SIGMA[j,j])
header = f"{'':>8}" + "".join(f"{t:>8}" for t in TICKERS)
print(header)
for i, t in enumerate(TICKERS):
    row = f"  {t:>6}" + "".join(f"{corr[i,j]:>8.3f}" for j in range(N))
    print(row)


def build_qubo(lam: float, lam_pen: float, K: int) -> np.ndarray:
    """
    Constructs the QUBO matrix Q for portfolio optimization.

    Derivation:
        Objective: λ x^T Σ x - μ^T x
        Penalty:   λ_pen (Σ x_i - K)^2 = λ_pen (x^T 1 1^T x - 2K 1^T x + K^2)

        QUBO matrix Q (upper triangular convention, then symmetrised):
          Q_ii = λ Σ_ii - μ_i + λ_pen(1 - 2K)
          Q_ij = λ Σ_ij + 2 λ_pen   (i ≠ j)

    Args:
        lam:     Risk-aversion parameter.
        lam_pen: Budget penalty weight.
        K:       Target number of assets.

    Returns:
        n×n QUBO matrix Q.
    """
    Q = lam * SIGMA - np.diag(MU)
    penalty_diag = lam_pen * (1 - 2*K)
    penalty_off  = 2 * lam_pen
    Q += penalty_diag * np.eye(N) + penalty_off * (np.ones((N,N)) - np.eye(N))
    return Q


def portfolio_cost(x: np.ndarray, Q: np.ndarray) -> float:
    """Evaluate QUBO objective x^T Q x."""
    return float(x @ Q @ x)


def portfolio_stats(x: np.ndarray):
    """Compute expected return and risk (std dev) of an equal-weight portfolio."""
    n_held   = x.sum()
    w        = x / n_held if n_held > 0 else x
    ret      = float(w @ MU)
    variance = float(w @ SIGMA @ w)
    return {'return': ret, 'risk': math.sqrt(variance), 'variance': variance}


# ── QUBO matrix ─────────────────────────────────────────────────────────────
Q = build_qubo(LAM, LAM_PEN, K)
print(f"\nQUBO Matrix Q (λ={LAM}, λ_pen={LAM_PEN}, K={K}):")
header2 = f"{'':>8}" + "".join(f"{t:>10}" for t in TICKERS)
print(header2)
for i, t in enumerate(TICKERS):
    row = f"  {t:>6}" + "".join(f"{Q[i,j]:>10.4f}" for j in range(N))
    print(row)

# ── Brute-force enumeration ──────────────────────────────────────────────────
print(f"\nBrute-Force Portfolio Ranking (K={K} assets):")
print("-" * 72)
print(f"{'Rank':>5}  {'Portfolio':>15}  {'Bitstring':>10}  "
      f"{'Cost':>10}  {'Return':>8}  {'Risk':>8}")
print("-" * 72)

candidates = []
for combo in itertools.combinations(range(N), K):
    x = np.array([1 if i in combo else 0 for i in range(N)])
    bits = ''.join(map(str, x))
    cost = portfolio_cost(x, Q)
    stats = portfolio_stats(x)
    name  = '+'.join(TICKERS[i] for i in combo)
    candidates.append((cost, name, bits, stats))

candidates.sort(key=lambda c: c[0])
for rank, (cost, name, bits, stats) in enumerate(candidates):
    marker = " ← OPTIMAL" if rank == 0 else ""
    print(f"  {rank+1:>3}   {name:>15}  {bits:>10}  "
          f"{cost:>10.4f}  {stats['return']*100:>6.1f}%  {stats['risk']*100:>6.1f}%"
          f"{marker}")

optimal_name = candidates[0][1]
optimal_cost = candidates[0][0]

# ── Efficient frontier plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#e53935','#1565c0','#f9ab00','#6a1b9a','#00838f','#558b2f']

for rank, (cost, name, bits, stats) in enumerate(candidates):
    marker = '*' if rank == 0 else 'o'
    size   = 200 if rank == 0 else 80
    label  = f"{name} (rank {rank+1})"
    axes[0].scatter(stats['risk']*100, stats['return']*100,
                    c=colors[rank], s=size, marker=marker, zorder=5, label=label)
    axes[0].annotate(name, (stats['risk']*100+0.2, stats['return']*100),
                     fontsize=8)

axes[0].set_xlabel('Portfolio Risk σ (%)')
axes[0].set_ylabel('Expected Return μ (%)')
axes[0].set_title('Efficient Frontier — 2-Asset Portfolios', fontsize=12)
axes[0].legend(fontsize=7, loc='lower right')
axes[0].grid(alpha=0.3)

# Bar chart of costs
names_sorted = [c[1] for c in candidates]
costs_sorted = [c[0] for c in candidates]
bar_colors   = [colors[i] for i in range(len(candidates))]
bars = axes[1].barh(names_sorted[::-1], costs_sorted[::-1],
                    color=bar_colors[::-1], edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('QUBO Cost $x^T Q x$')
axes[1].set_title(f'Portfolio Costs (λ={LAM})', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')
for bar, (cost,*_) in zip(bars[::-1], candidates):
    axes[1].text(cost - 0.005, bar.get_y() + bar.get_height()/2,
                 f'{cost:.3f}', va='center', ha='right',
                 fontsize=8, color='white', fontweight='bold')

plt.suptitle(f'Portfolio Optimization — {" · ".join(TICKERS)}', fontsize=13)
plt.tight_layout()
plt.savefig('portfolio_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nClassical optimal: {optimal_name}  (cost = {optimal_cost:.4f})")

---
## Exercise 2: QAOA Portfolio Optimization

We now map the QUBO to a quantum Hamiltonian and run QAOA. The substitution $x_i = (1-Z_i)/2$ converts $x^T Q x$ into:

$$H_\text{portfolio} = \sum_{i<j} \frac{Q_{ij}}{2} Z_i Z_j + \sum_i h_i Z_i + \text{const.}$$

where $h_i = -\frac{1}{2}\sum_j Q_{ij}$.

**Why is this the same circuit as Lab 4?**  
The QAOA circuit implements $e^{-i\gamma H_C}$ (Oracle) followed by $e^{-i\beta H_B}$ (Mixer). The Mixer $H_B = \sum_i X_i$ is always the same. The Oracle decomposes into:
- One CNOT–$R_z(2\gamma Q_{ij}/2)$–CNOT per $Z_iZ_j$ term  
- One $R_z(2\gamma h_i)$ per $Z_i$ term

The **only** difference from MaxCut is that the $R_z$ angles are now proportional to the financial coefficients $Q_{ij}$ instead of $\pm 1$.

**Why use COBYLA?**  
COBYLA is gradient-free — it does not require the parameter-shift rule. This is useful for circuits with many parameters ($2p$ in total) where each PSR gradient evaluation costs $2 \times 2p$ circuit runs. COBYLA needs fewer total evaluations for small $p$. For large $p$ or noisy hardware, gradient-based methods (SPSA, Adam with PSR) become competitive.

**What to observe:** The optimal portfolio $|0011\rangle$ (AAPL+MSFT in little-endian) should have the highest probability. Infeasible states (not exactly 2 ones) should be suppressed near zero by the penalty term.

**Student Experiments:**
1. Run with `P = 1`. Does the quantum output match the classical brute-force?
2. Change `P = 2`. Does the optimal state gain more probability?
3. Go back to Exercise 1 and set `LAM = 5.0`, then re-run both exercises. Does the optimal portfolio change? Does QAOA follow the shift?


In [ ]:
"""
QAOA Portfolio Optimization Module.
Maps the QUBO to a quantum Hamiltonian and runs QAOA with p ∈ {1,2} layers.
Compares the quantum result against the brute-force classical baseline.
"""

# ── STUDENT EXPERIMENT ZONE ────────────────────────────────────────────────
P        = 1      # Number of QAOA layers (try 2)
N_SHOTS  = 4000   # Measurement shots for sampling
# ──────────────────────────────────────────────────────────────────────────


def build_portfolio_hamiltonian(Q: np.ndarray) -> SparsePauliOp:
    """
    Converts a QUBO matrix Q to an Ising Hamiltonian H_portfolio.

    Mapping: x_i = (1 - Z_i) / 2
    x^T Q x = Σ_{i<j} (Q_ij/2) Z_i Z_j + Σ_i h_i Z_i + const
    where h_i = -(1/2) Σ_j Q_ij  (sum of i-th row / column)

    Qiskit convention: Pauli string 'XYZZ' acts as Z on q0, Z on q1,
    Y on q2, X on q3 (little-endian). Reverse the index accordingly.

    Args:
        Q: n×n symmetric QUBO matrix.

    Returns:
        SparsePauliOp representing H_portfolio (up to a constant).
    """
    n     = Q.shape[0]
    terms = []

    # ZZ terms from off-diagonal Q_ij
    for i in range(n):
        for j in range(i + 1, n):
            coeff = Q[i, j] / 2
            if abs(coeff) < 1e-12:
                continue
            zz = list('I' * n)
            zz[i] = 'Z'
            zz[j] = 'Z'
            zz.reverse()   # Qiskit little-endian
            terms.append((''.join(zz), coeff))

    # Z terms from row sums
    for i in range(n):
        h_i = -0.5 * Q[i, :].sum()
        if abs(h_i) < 1e-12:
            continue
        z = list('I' * n)
        z[i] = 'Z'
        z.reverse()
        terms.append((''.join(z), h_i))

    return SparsePauliOp.from_list(terms).simplify()


def build_qaoa_circuit(n: int, edges_zz: list, p: int) -> tuple:
    """
    Constructs a QAOA circuit for a general Ising Hamiltonian.

    Args:
        n:        Number of qubits.
        edges_zz: List of (i, j, coeff) for ZZ terms in H_C.
        p:        Number of QAOA layers.

    Returns:
        (circuit, parameter_list)
    """
    g  = ParameterVector('\u03b3', p)
    b  = ParameterVector('\u03b2', p)
    qc = QuantumCircuit(n, name=f'Portfolio QAOA p={p}')
    qc.h(range(n))
    qc.barrier()

    for k in range(p):
        # Cost unitary: e^{-iγ H_C}
        for (i, j, coeff) in edges_zz:
            # e^{-iγ coeff ZiZj} via CNOT – Rz(2γ coeff) – CNOT
            qc.cx(i, j)
            qc.rz(2 * g[k] * coeff, j)
            qc.cx(i, j)
        qc.barrier()
        # Mixer: e^{-iβ Σ Xi} = ⊗ Rx(2β)
        qc.rx(2 * b[k], range(n))
        qc.barrier()

    return qc, list(g) + list(b)


def extract_zz_terms(H: SparsePauliOp) -> list:
    """
    Extract ZZ terms from a SparsePauliOp for QAOA circuit construction.
    Returns list of (qubit_i, qubit_j, coefficient).
    Note: Pauli strings are stored in reversed (little-endian) order.
    """
    n      = H.num_qubits
    terms  = []
    for pauli, coeff in zip(H.paulis, H.coeffs):
        z_indices = [i for i in range(n) if str(pauli)[n-1-i] == 'Z']
        if len(z_indices) == 2:
            i, j = z_indices[0], z_indices[1]
            terms.append((i, j, float(coeff.real)))
    return terms


# ── Build Hamiltonian and circuit ─────────────────────────────────────────
H_port   = build_portfolio_hamiltonian(Q)
zz_terms = extract_zz_terms(H_port)

print(f"Portfolio Hamiltonian ({len(H_port)} Pauli terms):")
for pauli, coeff in zip(H_port.paulis, H_port.coeffs):
    if abs(coeff) > 0.01:
        print(f"  {str(pauli):>8}  {coeff.real:+.4f}")

qaoa_circ, params_list = build_qaoa_circuit(N, zz_terms, P)
display(qaoa_circ.draw('mpl', style='iqp', fold=50))
print(f"\nQAOA circuit: {N} qubits, p={P}, {len(params_list)} parameters")
print(f"Gate count: {qaoa_circ.count_ops()}")

# ── Classical optimisation ─────────────────────────────────────────────────
estimator  = Estimator()
cost_hist  = []

def cost_function(params):
    job = estimator.run([(qaoa_circ, H_port, params)])
    val = float(job.result()[0].data.evs)
    cost_hist.append(val)
    return val   # minimise ⟨H_portfolio⟩

# Multiple restarts
best_res, best_hist = None, None
for trial in range(5):
    cost_hist = []
    x0  = np.random.uniform(0, math.pi, len(params_list))
    res = minimize(cost_function, x0, method='COBYLA',
                   options={'maxiter': 400*P, 'rhobeg': 0.5})
    if best_res is None or res.fun < best_res.fun:
        best_res, best_hist = res, cost_hist.copy()
    print(f"  Trial {trial+1}: E_min = {res.fun:.4f}  iters = {res.nit}")

print(f"\nBest ⟨H_portfolio⟩ = {best_res.fun:.4f}")
print(f"Optimal parameters: γ={best_res.x[:P].round(3)}, β={best_res.x[P:].round(3)}")

# ── Sample the optimal circuit ─────────────────────────────────────────────
opt_circ  = qaoa_circ.assign_parameters(dict(zip(params_list, best_res.x)))
meas_circ = opt_circ.copy()
meas_circ.measure_all()
sim    = AerSimulator()
counts = sim.run(meas_circ, shots=N_SHOTS).result().get_counts()

print(f"\nTop measurement results ({N_SHOTS} shots):")
print("-" * 55)

# Decode bitstrings (Qiskit output: q_{n-1}...q_1 q_0)
FEASIBLE_BITS = set()
for combo in itertools.combinations(range(N), K):
    x    = [1 if i in combo else 0 for i in range(N)]
    bits = ''.join(map(str, x[::-1]))   # Qiskit: reversed
    FEASIBLE_BITS.add(bits)

sorted_counts = sorted(counts.items(), key=lambda x: -x[1])
for state, cnt in sorted_counts[:8]:
    prob  = cnt / N_SHOTS * 100
    # decode: Qiskit string is q3q2q1q0 → reverse for x=[q0,q1,q2,q3]
    x_arr = np.array([int(b) for b in state[::-1]])
    if x_arr.sum() == K:
        name = '+'.join(TICKERS[i] for i in range(N) if x_arr[i]==1)
        feas_tag = f"[{name}]"
        bf_cost  = portfolio_cost(x_arr, Q)
    else:
        feas_tag = '[infeasible]'
        bf_cost  = float('nan')
    mark = " ← optimal" if feas_tag == f'[{optimal_name}]' else ''
    print(f"  |{state}⟩  {cnt:5d} shots  ({prob:5.1f}%)  "
          f"{feas_tag:>20}  cost={bf_cost:.4f}{mark}")

display(plot_histogram(counts, title=f"QAOA Portfolio (p={P})",
                       figsize=(10, 4)))

---
## Exercise 3: Business Analysis and Result Interpretation

The QAOA output is a **probability distribution**, not a single answer. This is a feature, not a bug: it gives you a ranked list of portfolio candidates, each with a probability that reflects how close it is to optimal.

**How to read the output:**
- The most probable **feasible** bitstring (exactly 2 ones) is the primary recommendation.
- The second and third most probable feasible states are secondary recommendations — useful for diversification across model uncertainty.
- Infeasible states (wrong number of assets) should appear with near-zero probability; if they do not, increase $\lambda_\text{pen}$ or $p$.

**Why is the second-best portfolio useful in practice?**  
Financial models have parameter uncertainty — the covariance matrix $\Sigma$ and expected returns $\mu$ are estimates from historical data. A portfolio that is optimal under the model may underperform if the estimates are off. The second-best portfolio often has a different risk profile and serves as a hedge against model error. A classical solver gives you one answer; QAOA gives you a ranked shortlist.

**What this cell computes:**
1. The top-3 most probable feasible portfolios from the QAOA sample.
2. Their actual expected return, risk, and Sharpe ratio (computed from the asset data, not the QUBO).
3. The sub-optimality gap: how much worse (in QUBO cost) the second and third recommendations are compared to the optimal.


In [ ]:
"""
Business Analysis Module.
Extracts the top-3 feasible portfolios from QAOA output and
computes their financial characteristics.
"""

# ── Extract top-3 feasible from QAOA sample ────────────────────────────────
feasible_results = []
for state, cnt in sorted_counts:
    x_arr = np.array([int(b) for b in state[::-1]])
    if x_arr.sum() == K:
        name   = '+'.join(TICKERS[i] for i in range(N) if x_arr[i]==1)
        stats  = portfolio_stats(x_arr)
        bf_cost = portfolio_cost(x_arr, Q)
        feasible_results.append({
            'name':   name,
            'state':  state,
            'prob':   cnt/N_SHOTS,
            'return': stats['return'],
            'risk':   stats['risk'],
            'cost':   bf_cost
        })

top3 = sorted(feasible_results, key=lambda x: -x['prob'])[:3]

print("Business Analysis: Top-3 QAOA Portfolio Recommendations")
print("=" * 68)

for rank, p in enumerate(top3):
    is_opt  = p['name'] == optimal_name
    opt_tag = " ✓ CLASSICAL OPTIMUM" if is_opt else ""
    cost_gap = abs(p['cost'] - optimal_cost)

    print(f"\n  Rank {rank+1}: {p['name']}{opt_tag}")
    print(f"    QAOA probability:    {p['prob']*100:.1f}%")
    print(f"    Expected return:     {p['return']*100:.1f}% p.a.")
    print(f"    Portfolio risk (σ):  {p['risk']*100:.1f}% p.a.")
    print(f"    Sharpe (approx):     {p['return']/p['risk']:.2f}")
    print(f"    QUBO cost:           {p['cost']:.4f}")
    if not is_opt:
        print(f"    Sub-optimality gap:  {cost_gap:.4f} "
              f"({cost_gap/abs(optimal_cost)*100:.1f}% from optimal)")

# ── Comparison figure ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: risk-return scatter with QAOA probabilities as bubble size
all_feas = {r['name']:r for r in feasible_results}
colors   = ['#2e7d32','#1565c0','#f9ab00','#9c27b0','#e53935','#546e7a']

for idx, (cost, name, bits, stats) in enumerate(candidates):
    r   = all_feas.get(name, {}).get('prob', 0.01)
    sz  = max(r * 3000, 30)
    is_opt = name == optimal_name
    ec  = 'gold' if is_opt else 'white'
    lw  = 3     if is_opt else 1
    axes[0].scatter(stats['risk']*100, stats['return']*100,
                    s=sz, c=colors[idx], edgecolors=ec,
                    linewidths=lw, zorder=5, alpha=0.85)
    axes[0].annotate(f"{name}\n({r*100:.0f}%)",
                     (stats['risk']*100+0.2, stats['return']*100),
                     fontsize=7.5)

axes[0].set_xlabel('Portfolio Risk σ (%)')
axes[0].set_ylabel('Expected Return (%)')
axes[0].set_title(f'Efficient Frontier\n(bubble size ∝ QAOA probability, p={P})',
                  fontsize=11)
axes[0].grid(alpha=0.3)
gold_patch  = mpatches.Patch(edgecolor='gold', facecolor='white', lw=2,
                              label='Classical optimum')
axes[0].legend(handles=[gold_patch], fontsize=8)

# Right: convergence + comparison table
axes[1].plot(best_hist, 'b-', lw=1.5, alpha=0.85, label=f'QAOA p={P}')
axes[1].axhline(optimal_cost, color='green', linestyle='--', lw=2,
                label=f'Classical optimum ({optimal_cost:.4f})')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('$\\langle H_\\mathrm{portfolio}\\rangle$')
axes[1].set_title(f'COBYLA Convergence — p={P}', fontsize=11)
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle(f'QAOA Portfolio Optimization — {" · ".join(TICKERS)}',
             fontsize=13)
plt.tight_layout()
plt.savefig('qaoa_portfolio_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
top1 = top3[0]
found_opt = top1['name'] == optimal_name
print(f"\n── Summary ──")
print(f"QAOA top recommendation: {top1['name']}  ({top1['prob']*100:.1f}% probability)")
print(f"Classical optimal:       {optimal_name}  (cost {optimal_cost:.4f})")
print(f"Match:                   {'✓ YES' if found_opt else '✗ NO — check p or re-run'}")

---
## OPTIONAL Exercise 4: Scaling to n = 6 Assets

We add GOOGL and AMZN to the universe. With $n=6$, $K=3$, there are
$\binom{6}{3} = 20$ possible portfolios — still brute-forceable classically but approaching
the regime where classical enumeration begins to slow.

**Why does this exercise matter?**  
The $n=4$ case is pedagogically clean but not practically interesting — any laptop solves it instantly. The $n=6$ case begins to show the circuit scaling properties of QAOA:
- The number of ZZ terms grows as $\binom{n}{2}$: from 6 (n=4) to 15 (n=6).
- The circuit depth per layer grows from $\mathcal{O}(n)$ to $\mathcal{O}(n^2)$.
- The classical brute-force grows from $\binom{4}{2}=6$ to $\binom{6}{3}=20$ — manageable.
- At $n=50$ the brute-force reaches $\sim 10^{10}$ — intractable. QAOA's $\binom{50}{2}=1225$ ZZ terms per layer is still manageable.

**What to observe:** The QAOA approximation ratio for $n=6$, $p=1$ is lower than for $n=4$, $p=1$. This is expected: more states to discriminate with the same number of parameters. Increasing $p$ recovers accuracy at the cost of circuit depth.


In [ ]:
"""
Scaling Module (Optional).
Extends the portfolio universe to n=6 assets (K=3)
and compares circuit complexity against the n=4 case.
"""

# 6-asset universe: AAPL, MSFT, JPM, XOM + GOOGL, AMZN
TICKERS_6 = ['AAPL', 'MSFT', 'JPM', 'XOM', 'GOOGL', 'AMZN']
N6, K6    = 6, 3

MU_6 = np.array([0.340, 0.270, 0.150, 0.220, 0.260, 0.300])

# Extended covariance (approximate, 2020-2024)
SIGMA_6 = np.array([
    [0.0841, 0.0588, 0.0313, 0.0203, 0.0620, 0.0550],
    [0.0588, 0.0729, 0.0272, 0.0170, 0.0580, 0.0510],
    [0.0313, 0.0272, 0.0576, 0.0294, 0.0260, 0.0240],
    [0.0203, 0.0170, 0.0294, 0.1225, 0.0180, 0.0190],
    [0.0620, 0.0580, 0.0260, 0.0180, 0.0784, 0.0650],
    [0.0550, 0.0510, 0.0240, 0.0190, 0.0650, 0.0900]
])

def build_qubo_general(mu, sigma, lam=1.0, lam_pen=5.0, K=3):
    n = len(mu)
    Q = lam * sigma - np.diag(mu)
    Q += lam_pen * (1-2*K) * np.eye(n)
    Q += lam_pen * 2       * (np.ones((n,n)) - np.eye(n))
    return Q

Q6 = build_qubo_general(MU_6, SIGMA_6, lam=1.0, lam_pen=5.0, K=K6)
H6 = build_portfolio_hamiltonian(Q6)
zz6 = extract_zz_terms(H6)

qaoa6, params6 = build_qaoa_circuit(N6, zz6, p=1)

print("Scaling Comparison: n=4 vs n=6")
print("-" * 50)
print(f"  n=4: {qaoa_circ.count_ops()['cx']:>4} CX gates, "
      f"{qaoa_circ.count_ops()['rz']:>3} Rz,  "
      f"{qaoa_circ.count_ops()['rx']:>2} Rx")
print(f"  n=6: {qaoa6.count_ops()['cx']:>4} CX gates, "
      f"{qaoa6.count_ops()['rz']:>3} Rz,  "
      f"{qaoa6.count_ops()['rx']:>2} Rx")
print(f"  n=4 feasible portfolios: C(4,2) = {math.comb(4,2)}")
print(f"  n=6 feasible portfolios: C(6,3) = {math.comb(6,3)}")
print(f"  ZZ terms n=4: C(4,2)={math.comb(4,2)} | ZZ terms n=6: C(6,2)={math.comb(6,2)}")
print(f"  Circuit depth grows O(n²): {math.comb(4,2)} → {math.comb(6,2)} = "
      f"×{math.comb(6,2)/math.comb(4,2):.1f}")

# Brute force for n=6
print(f"\nBrute-force C(6,3)={math.comb(6,3)} portfolios:")
cands6 = []
for combo in itertools.combinations(range(N6), K6):
    x   = np.array([1 if i in combo else 0 for i in range(N6)])
    name = '+'.join(TICKERS_6[i] for i in combo)
    w    = x / K6
    ret  = float(w @ MU_6)
    var  = float(w @ SIGMA_6 @ w)
    cost = float(x @ Q6 @ x)
    cands6.append((cost, name, ret, math.sqrt(var)))
cands6.sort()
for rank, (cost, name, ret, risk) in enumerate(cands6[:5]):
    mark = " ← OPTIMAL" if rank==0 else ""
    print(f"  {rank+1}. {name:<25}  cost={cost:.4f}  "
          f"ret={ret*100:.1f}%  risk={risk*100:.1f}%{mark}")

# Run QAOA p=1 on n=6
estimator6 = Estimator()
cost_hist6 = []

def cost6(params):
    job = estimator6.run([(qaoa6, H6, params)])
    val = float(job.result()[0].data.evs)
    cost_hist6.append(val)
    return val

x0_6  = np.random.uniform(0, math.pi, len(params6))
res6  = minimize(cost6, x0_6, method='COBYLA', options={'maxiter':500})

opt6  = qaoa6.assign_parameters(dict(zip(params6, res6.x)))
opt6.measure_all()
cnt6 = sim.run(opt6, shots=4000).result().get_counts()

print(f"\nQAOA n=6 result (p=1):")
for state, cnt in sorted(cnt6.items(), key=lambda x: -x[1])[:5]:
    x_arr = np.array([int(b) for b in state[::-1]])
    feas  = x_arr.sum() == K6
    name  = '+'.join(TICKERS_6[i] for i in range(N6) if x_arr[i]==1) if feas else 'infeasible'
    prob  = cnt/4000*100
    mark  = " ← optimal" if feas and name == cands6[0][1] else ""
    print(f"  |{state}⟩  {prob:5.1f}%  {name}{mark}")

---
## OPTIONAL Exercise 5: Sensitivity Analysis — Risk Aversion $\lambda$

This exercise sweeps $\lambda \in \{0.5, 1.0, 1.5, 2.0, 3.0, 5.0\}$ and identifies which portfolio is optimal at each value.

**Why does $\lambda$ shift the optimal portfolio?**  
The QUBO objective balances two terms:
- **Risk term** $\lambda x^T\Sigma x$: penalises portfolios with high return variance. Weight $\lambda$.
- **Return term** $-\mu^T x$: rewards portfolios with high expected return. Weight 1.

At small $\lambda$ (aggressive investor), return dominates: AAPL+MSFT win because they have the highest $\mu$ values, even though they are highly correlated (tech sector).

At large $\lambda$ (defensive investor), variance dominates: the optimizer prefers JPM+XOM because they have the lowest covariance ($\Sigma_{\text{JPM,XOM}} = 0.0294$, the smallest off-diagonal entry) — even though their returns are modest.

**Business interpretation:** This exercise traces the **quantum efficient frontier** — the set of QAOA-recommended portfolios for different investor risk preferences. In a real application you would run this sweep for a client, show them the frontier, and let them choose their $\lambda$ based on their risk tolerance.


In [ ]:
"""
Sensitivity Analysis Module (Optional).
Computes the optimal portfolio for several values of λ and plots
the resulting quantum efficient frontier.
"""

lambdas = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
frontier_points = []

print("Lambda Sensitivity Analysis — Classical Brute-Force")
print("-" * 60)
print(f"{'λ':>5}  {'Optimal Portfolio':>18}  {'Return':>8}  {'Risk':>8}  {'Cost':>10}")
print("-" * 60)

prev_opt = None
for lam in lambdas:
    Q_lam = build_qubo(lam, LAM_PEN, K)
    cands = []
    for combo in itertools.combinations(range(N), K):
        x    = np.array([1 if i in combo else 0 for i in range(N)])
        name = '+'.join(TICKERS[i] for i in combo)
        cost = portfolio_cost(x, Q_lam)
        stats = portfolio_stats(x)
        cands.append((cost, name, stats))
    cands.sort()
    opt_cost, opt_name, opt_stats = cands[0]
    changed = '← changed!' if opt_name != prev_opt and prev_opt is not None else ''
    print(f"  {lam:>4.1f}  {opt_name:>18}  {opt_stats['return']*100:>6.1f}%  "
          f"{opt_stats['risk']*100:>6.1f}%  {opt_cost:>10.4f}  {changed}")
    frontier_points.append({
        'lam':    lam,
        'name':   opt_name,
        'return': opt_stats['return'],
        'risk':   opt_stats['risk']
    })
    prev_opt = opt_name

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: lambda vs optimal portfolio
unique_names = list(dict.fromkeys(p['name'] for p in frontier_points))
colors_map   = {n: c for n, c in zip(unique_names,
                ['#e53935','#1565c0','#f9ab00','#6a1b9a','#00838f'])}

for p in frontier_points:
    axes[0].scatter(p['lam'], p['return']*100,
                    c=colors_map[p['name']], s=120, zorder=5)

axes[0].set_xlabel('Risk Aversion λ')
axes[0].set_ylabel('Optimal Portfolio Return (%)')
axes[0].set_title('Optimal Return vs Risk Aversion', fontsize=11)
legend_patches = [mpatches.Patch(color=c, label=n)
                  for n,c in colors_map.items()]
axes[0].legend(handles=legend_patches, fontsize=8)
axes[0].grid(alpha=0.3)

# Right: efficient frontier — risk vs return for all portfolios, highlight optimal at each lambda
all_stats = []
for combo in itertools.combinations(range(N), K):
    x     = np.array([1 if i in combo else 0 for i in range(N)])
    name  = '+'.join(TICKERS[i] for i in combo)
    stats = portfolio_stats(x)
    all_stats.append({'name':name, 'return':stats['return'], 'risk':stats['risk']})

for idx, s in enumerate(all_stats):
    axes[1].scatter(s['risk']*100, s['return']*100,
                    c='#90a4ae', s=60, zorder=3, alpha=0.7)
    axes[1].annotate(s['name'], (s['risk']*100+0.15, s['return']*100),
                     fontsize=7, color='#607d8b')

for fp in frontier_points:
    s = next(s for s in all_stats if s['name']==fp['name'])
    axes[1].scatter(s['risk']*100, s['return']*100,
                    c=colors_map[fp['name']], s=140, zorder=5,
                    edgecolors='black', linewidths=1)

axes[1].set_xlabel('Portfolio Risk σ (%)')
axes[1].set_ylabel('Expected Return (%)')
axes[1].set_title('Efficient Frontier\n(coloured = optimal for some λ)', fontsize=11)
axes[1].legend(handles=legend_patches, fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle('Risk-Aversion Sensitivity: How λ Shifts the Optimal Portfolio',
             fontsize=13)
plt.tight_layout()
plt.savefig('lambda_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()